# A2: band histograms, EuroSAT against Thailand

Does a pixel of the same class have the same reflectance in Europe and in Thailand? One histogram per band, three
sources drawn on top of each other:

| line | data | role |
|---|---|---|
| EuroSAT (train) | `eurosat_X.npy`, training patches only | what the model learns from |
| Thai L1C | `thailand_X_L1C.npy`, pure patches | the main test set (same processing level as EuroSAT) |
| Thai L2A | `thailand_X_L2A.npy`, same patches | the comparison: does the processing level matter? |

**Why per class, and why balanced.** EuroSAT has most built-up patches, the Thai test set has most forest. If all patches were pooled,
a difference could just mean "different mix of classes". So the same number of pixels is drawn from each of the four evaluated classes
(forest, crop, built-up, water) on both sides, and every class also gets its own figure. Grassland is left out: no Thai patch passes the purity threshold.

**Things to keep in mind when you read the figures**
- The Thai images are a January-February median composite (dry season); every EuroSAT patch is a single date from any season. Part of a difference can come from this, not from the region.
- Each Thai class comes from one or two study areas (see `DATA_NOTES.md`), so a Thai class stands for that area, not for the whole country.
- Reflectance is stored as reflectance x 10000.

In [ ]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.append(str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import config as c

# chart colours on a light surface: categorical slots 1-3 of the data-viz palette (checked with its validator)
SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
COLORS = {"EuroSAT (train)": "#2a78d6", "Thai L1C": "#eb6834", "Thai L2A": "#1baf7a"}

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.edgecolor": AXIS, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "x", "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "font.family": "sans-serif", "font.size": 10,
})

BAND_NAMES = {"B1": "coastal aerosol", "B2": "blue", "B3": "green", "B4": "red", "B5": "red edge 1", "B6": "red edge 2",
              "B7": "red edge 3", "B8": "near infrared", "B8A": "narrow NIR", "B9": "water vapour",
              "B11": "SWIR 1", "B12": "SWIR 2"}
OUT = c.OUTPUT_DIR / "a2"
OUT.mkdir(parents=True, exist_ok=True)


def save(fig, name):
    # Windows sometimes refuses to open a file that was written a moment ago, so retry a few times
    path = OUT / name
    for attempt in range(5):
        try:
            fig.savefig(path, dpi=130, bbox_inches="tight")
            break
        except OSError:
            if attempt == 4:
                raise
            time.sleep(1)
    print("saved", path)

## 1. Load the data

The arrays are opened with `mmap_mode="r"`, so only the patches that are used are read from disk. The CSV is read with
`keep_default_na=False`, otherwise the area name `nan` (Nan province) becomes an empty value.

In [ ]:
EU_X = np.load(c.EUROSAT_DIR / "eurosat_X.npy", mmap_mode="r")      # (27000, 12, 64, 64)
EU_Y = np.load(c.EUROSAT_DIR / "eurosat_y.npy")                      # 5 target classes
EU_SPLIT = np.load(c.EUROSAT_DIR / "eurosat_split.npy")              # 0 train, 1 val, 2 test
TH_X = {"Thai L1C": np.load(c.PATCH_DIR / "thailand_X_L1C.npy", mmap_mode="r"),
        "Thai L2A": np.load(c.PATCH_DIR / "thailand_X_L2A.npy", mmap_mode="r")}
TH = pd.read_csv(c.PATCH_DIR / "thailand_patches.csv", keep_default_na=False)

EVAL = c.THAI_EVAL_CLASSES      # forest, crop, built-up, water
rows = []
for cls in EVAL:
    rows.append({"class": cls,
                 "EuroSAT train patches": int(((EU_Y == c.CLASSES.index(cls)) & (EU_SPLIT == 0)).sum()),
                 "Thai pure patches": int((TH["pure"] & (TH["label"] == cls)).sum())})
pd.DataFrame(rows).set_index("class")

## 2. Draw the pixels

For every class: up to 300 random patches, then 100,000 random pixels out of them (seed 42). The two Thai products use the **same patches
and the same pixels**, so L1C and L2A can be compared one to one.

In [ ]:
PATCHES_PER_CLASS = 300
PIXELS_PER_CLASS = 100_000


def draw(patch_ids, rng):
    # which patches, and which pixel positions inside the 300 x 4096 pixels of those patches
    ids = np.sort(rng.choice(patch_ids, size=min(PATCHES_PER_CLASS, len(patch_ids)), replace=False))
    pix = np.sort(rng.choice(len(ids) * 64 * 64, size=min(PIXELS_PER_CLASS, len(ids) * 64 * 64), replace=False))
    return ids, pix


def take(X, ids, pix):
    block = np.asarray(X[ids])                                  # (patches, 12, 64, 64)
    return block.transpose(0, 2, 3, 1).reshape(-1, 12)[pix].astype("float32")


rng = np.random.default_rng(c.SEED)
samples = {name: {} for name in COLORS}
for cls in EVAL:
    eu_ids = np.flatnonzero((EU_Y == c.CLASSES.index(cls)) & (EU_SPLIT == 0))
    ids, pix = draw(eu_ids, rng)
    samples["EuroSAT (train)"][cls] = take(EU_X, ids, pix)
    th_ids = TH.loc[TH["pure"] & (TH["label"] == cls), "patch_id"].to_numpy()
    ids, pix = draw(th_ids, rng)
    for name, X in TH_X.items():
        samples[name][cls] = take(X, ids, pix)

# the same number of pixels from every class: the class mix is then equal on both sides
balanced = {name: np.concatenate([samples[name][cls] for cls in EVAL]) for name in COLORS}
print({name: arr.shape for name, arr in balanced.items()})

## 3. Histograms, all four classes in equal parts

Each panel is one band. A curve that sits to the right of another means higher reflectance in that source. The vertical axis is the share of
pixels per value (density), so sources with different numbers of pixels can be compared; its numbers carry no meaning, so they are hidden.
The horizontal range stops at the 0.5th and 99.5th percentile of the three sources together.

In [ ]:
def band_grid(data, title, filename):
    fig, axes = plt.subplots(3, 4, figsize=(15, 9.2))
    for b, ax in enumerate(axes.ravel()):
        everything = np.concatenate([d[:, b] for d in data.values()])
        lo, hi = np.percentile(everything, (0.5, 99.5))
        bins = np.linspace(lo, hi, 70)
        for name, d in data.items():
            ax.hist(d[:, b], bins=bins, density=True, histtype="step", linewidth=2, color=COLORS[name])
        ax.set_title(f"{c.BANDS[b]}  {BAND_NAMES[c.BANDS[b]]}", loc="left", fontsize=10, color=INK)
        ax.set_yticks([])
        if b >= 8:
            ax.set_xlabel("reflectance x 10000")
    handles = [Line2D([0], [0], color=col, linewidth=2.5, label=name) for name, col in COLORS.items()]
    fig.legend(handles=handles, loc="upper center", ncol=3, frameon=False, fontsize=11, bbox_to_anchor=(0.5, 1.0))
    fig.suptitle(title, y=1.04, fontsize=13, color=INK)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    save(fig, filename)
    plt.show()


band_grid(balanced, "All bands: forest, crop, built-up and water in equal parts", "histograms_balanced.png")

## 4. The same, one class at a time

If a band differs between the regions for one class but not for another, the shift depends on the class.

In [ ]:
for cls in EVAL:
    band_grid({name: samples[name][cls] for name in COLORS}, f"{cls}: pixels of the same class", f"histograms_{cls}.png")

## 5. The numbers behind the figures

Median, mean, standard deviation and the 5th and 95th percentile of every band, for every class and for the balanced mix. Saved as
`outputs/a2/band_stats.csv`; step A4 (the size of the domain shift per band) starts from this table.

In [ ]:
records = []
groups = {**{cls: {name: samples[name][cls] for name in COLORS} for cls in EVAL}, "balanced mix": balanced}
for group, data in groups.items():
    for name, arr in data.items():
        for b, band in enumerate(c.BANDS):
            v = arr[:, b]
            records.append({"group": group, "source": name, "band": band, "mean": v.mean(), "median": np.median(v),
                            "std": v.std(), "p05": np.percentile(v, 5), "p95": np.percentile(v, 95)})
stats = pd.DataFrame(records)
stats.round(1).to_csv(OUT / "band_stats.csv", index=False)

# median of every band in the balanced mix, and how far each Thai product is from EuroSAT
mix = stats[stats["group"] == "balanced mix"].pivot(index="band", columns="source", values="median").reindex(c.BANDS)
mix["Thai L1C - EuroSAT"] = mix["Thai L1C"] - mix["EuroSAT (train)"]
mix["Thai L2A - EuroSAT"] = mix["Thai L2A"] - mix["EuroSAT (train)"]
mix.round(0).astype(int)

## 6. Questions to answer for the slide

- In which bands do the Thai curves sit clearly apart from EuroSAT, and in which do they overlap?
- Does the shift look alike in all four classes, or is it much larger for some (which ones)?
- Is Thai L1C or Thai L2A closer to EuroSAT? (EuroSAT is L1C, so a closer L1C is what the processing level predicts.)
- Which differences could come from the January-February composite and not from the region?